# A112509 — Results Comparison ($n = 80$ to $200$)

Compares the best values found by three independent methods for each $n$:

| Column | Source | Directory |
|--------|--------|-----------|
| **Unbounded MH** | Metropolis-Hastings, no structural constraints | `results/mh_unbounded/` |
| **Bounded MH** | Metropolis-Hastings, structural constraints applied | `results/mh_bounded/` |
| **Exhaustive** | Block-template exhaustive search (exact) | `data/cached_results.json` |

A ✓ in the **Agree** column means all available values for that $n$ match.  
A ✗ means at least two sources disagree (possible bound miss or MH shortfall).  
A — means data is only available from one source (no cross-check possible).

In [ ]:

import json
import os
from IPython.display import display, HTML

BASE = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
N_START = 80
N_END   = 200

# ── Load all three data sources ───────────────────────────────────────────────

def load_mh_dir(folder):
    """Return dict {n: (best_value, num_solutions)} from a directory of n_XXXX_results.json files."""
    out = {}
    path = os.path.join(BASE, folder)
    if not os.path.isdir(path):
        return out
    for fname in sorted(os.listdir(path)):
        if not fname.startswith("n_") or not fname.endswith("_results.json"):
            continue
        try:
            n = int(fname[2:6])
        except ValueError:
            continue
        if n < N_START or n > N_END:
            continue
        with open(os.path.join(path, fname)) as f:
            data = json.load(f)
        val = data.get("best_value")
        if val is not None:
            sols = data.get("solutions", [])
            out[n] = (int(val), len(sols))
    return out

def load_cached():
    path = os.path.join(BASE, "data", "cached_results.json")
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        raw = json.load(f)
    return {int(k): (int(v["a(n)"]), int(v.get("num_optimal", 0)))
            for k, v in raw.items()
            if N_START <= int(k) <= N_END}

unbounded  = load_mh_dir("results/mh_unbounded")
bounded    = load_mh_dir("results/mh_bounded")
exhaustive = load_cached()

print(f"Unbounded MH entries (n={N_START}-{N_END}): {len(unbounded)}")
print(f"Bounded MH entries   (n={N_START}-{N_END}): {len(bounded)}")
print(f"Exhaustive entries   (n={N_START}-{N_END}): {len(exhaustive)}")

# ── Build and display the comparison table ────────────────────────────────────

def fmt_mh(mh_tuple, ex_tuple):
    if mh_tuple is None:
        return "<span style='color:#aaa'>-</span>"
    val, n_sols = mh_tuple
    cell = str(val)
    if n_sols is not None:
        if ex_tuple is not None and ex_tuple[1] > 0:
            pct = 100.0 * n_sols / ex_tuple[1]
            cell += f"<br><small style='color:#555'>({n_sols} sols / {pct:.0f}%)</small>"
        else:
            cell += f"<br><small style='color:#555'>({n_sols} sols)</small>"
    return cell

def fmt_exhaustive(ex_tuple):
    if ex_tuple is None:
        return "<span style='color:#aaa'>-</span>"
    val, n_sols = ex_tuple
    cell = str(val)
    if n_sols:
        cell += f"<br><small style='color:#555'>({n_sols} sols)</small>"
    return cell

def agree_cell(vals):
    present = [v for v in vals if v is not None]
    if len(present) < 2:
        return "<span style='color:#aaa'>-</span>"
    if len(set(present)) == 1:
        return "<span style='color:green; font-weight:bold'>OK</span>"
    return "<span style='color:red; font-weight:bold'>MISMATCH</span>"

def sol_coverage_cell(ub, bd, ex):
    """Check exhaustive solution count >= MH solution counts."""
    if ex is None:
        return "<span style='color:#aaa'>-</span>"
    ex_sols = ex[1]
    mh_counts = [t[1] for t in (ub, bd) if t is not None]
    if not mh_counts:
        return "<span style='color:#aaa'>-</span>"
    if all(ex_sols >= m for m in mh_counts):
        return "<span style='color:green; font-weight:bold'>✓</span>"
    worst = max(mh_counts)
    return f"<span style='color:red; font-weight:bold'>✗</span><br><small style='color:red'>ex={ex_sols} &lt; mh={worst}</small>"

# Pre-compute best value per n for delta calculation
best_val = {}
for n in range(N_START, N_END + 1):
    ub = unbounded.get(n)
    bd = bounded.get(n)
    ex = exhaustive.get(n)
    v = (ub[0] if ub else None) or (bd[0] if bd else None) or (ex[0] if ex else None)
    if v is not None:
        best_val[n] = v

rows = ""
for n in range(N_START, N_END + 1):
    ub = unbounded.get(n)
    bd = bounded.get(n)
    ex = exhaustive.get(n)
    if ub is None and bd is None and ex is None:
        continue
    ub_val = ub[0] if ub else None
    bd_val = bd[0] if bd else None
    ex_val = ex[0] if ex else None
    present_vals = [v for v in (ub_val, bd_val, ex_val) if v is not None]
    is_mismatch = len(present_vals) >= 2 and len(set(present_vals)) > 1
    row_bg = "#fff0f0" if is_mismatch else "#ffffff"

    cur = best_val.get(n)
    prev = best_val.get(n - 1)
    if cur is not None and prev is not None:
        delta_str = str(cur - prev)
    else:
        delta_str = "<span style='color:#aaa'>-</span>"

    rows += (
        f"<tr style='background:{row_bg}; color:#000;'>"
        f"<td style='text-align:right; padding:3px 10px; vertical-align:top;'>{n}</td>"
        f"<td style='text-align:right; padding:3px 10px; vertical-align:top;'>{fmt_mh(ub, ex)}</td>"
        f"<td style='text-align:right; padding:3px 10px; vertical-align:top;'>{fmt_mh(bd, ex)}</td>"
        f"<td style='text-align:right; padding:3px 10px; vertical-align:top;'>{fmt_exhaustive(ex)}</td>"
        f"<td style='text-align:center; padding:3px 10px; vertical-align:top;'>{agree_cell((ub_val, bd_val, ex_val))}</td>"
        f"<td style='text-align:right; padding:3px 10px; vertical-align:top;'>{delta_str}</td>"
        f"<td style='text-align:center; padding:3px 10px; vertical-align:top;'>{sol_coverage_cell(ub, bd, ex)}</td>"
        f"</tr>\n"
    )

html = f"""
<h3 style="color:#000;">A112509 Results Comparison: n = {N_START} to {N_END}</h3>
<table style="border-collapse:collapse; font-size:13px; color:#000;">
<thead>
<tr style="border-bottom:2px solid #333; background:#f5f5f5; color:#000;">
  <th style="padding:5px 10px; text-align:right; color:#000;">n</th>
  <th style="padding:5px 10px; text-align:right; color:#000;">Unbounded MH</th>
  <th style="padding:5px 10px; text-align:right; color:#000;">Bounded MH</th>
  <th style="padding:5px 10px; text-align:right; color:#000;">Exhaustive</th>
  <th style="padding:5px 10px; text-align:center; color:#000;">Agree on Optimal</th>
  <th style="padding:5px 10px; text-align:right; color:#000;">Δa(n)</th>
  <th style="padding:5px 10px; text-align:center; color:#000;">Exhaustive ≥ MH sols</th>
</tr>
</thead>
<tbody>
{rows}
</tbody>
</table>
<p style="font-size:12px; color:#555; margin-top:8px;">
  <b>Agree on Optimal:</b> OK = all available values agree | MISMATCH = disagreement (red row) | - = only one source<br>
  <b>Exhaustive ≥ MH sols:</b> ✓ = exhaustive solution count ≥ both MH counts | ✗ = exhaustive has fewer solutions than an MH run<br>
  Unbounded &amp; Bounded MH show: best value, then (# solutions found / % of exhaustive total)<br>
  Exhaustive shows: best value, then (# optimal solutions)<br>
  Δa(n) = a(n) − a(n−1) using best available value (unbounded MH preferred)
</p>
"""
display(HTML(html))

Unbounded MH entries (n=80-200): 121
Bounded MH entries   (n=80-200): 120
Exhaustive entries   (n=80-200): 71


n,Unbounded MH,Bounded MH,Exhaustive,Agree on Optimal,Δa(n),Exhaustive ≥ MH sols
80,2423(104 sols / 100%),-,2423(104 sols),OK,-,✓
81,2487(179 sols / 100%),2487(173 sols / 97%),2487(179 sols),OK,64,✓
82,2552(187 sols / 100%),2552(175 sols / 94%),2552(187 sols),OK,65,✓
83,2618(156 sols / 100%),2618(151 sols / 97%),2618(156 sols),OK,66,✓
84,2685(156 sols / 100%),2685(151 sols / 97%),2685(156 sols),OK,67,✓
85,2753(155 sols / 100%),2753(154 sols / 99%),2753(155 sols),OK,68,✓
86,2822(103 sols / 100%),2822(103 sols / 100%),2822(103 sols),OK,69,✓
87,2892(38 sols / 100%),2892(38 sols / 100%),2892(38 sols),OK,70,✓
88,2963(6 sols / 100%),2963(6 sols / 100%),2963(6 sols),OK,71,✓
89,3034(44 sols / 100%),3034(40 sols / 91%),3034(44 sols),OK,71,✓
